In [1]:
!pip install torch_optimizer
!pip install adan-pytorch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 2.9 MB/s eta 0:00:00


In [2]:
import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

import torch.optim as optim_std
import torch_optimizer as optim

from adan_pytorch import Adan

from tqdm import tqdm

In [3]:
df = pd.read_csv('weekly_driving_profiles.csv')

In [4]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

feature_cols = ['engine_capacity', 'road_quality_moderate', 'slope_flat',
                'motorway', 'rural', 'more_than_one_lane', 'congested',
                'speed_limit_mean', 'weather_temperature_mean', 'total_distance',
                'sum_roundabout', 'sum_traffic_signal', 'sum_stop_sign',
                'sum_yield_sign', 'sum_pedestrian_crossing_sign',
                'sum_animal_crossing_sign', 'speeding_serious',
                'harsh_acceleration', 'harsh_braking']
target_col = 'claims_count'

unique_drivers = df['driver_id'].unique()

#примерно 70% обучение 30% тест
train_drivers, test_drivers = train_test_split(unique_drivers, test_size=0.3, random_state=30)

X_train = df[df['driver_id'].isin(train_drivers)][feature_cols]
X_test = df[df['driver_id'].isin(test_drivers)][feature_cols]
y_train = df[df['driver_id'].isin(train_drivers)][target_col]
y_test = df[df['driver_id'].isin(test_drivers)][target_col]

print("Количество водителей в тренировке: ",len(train_drivers), "Количество строк: ",len(X_train))
print("Количество водителей в тесте: ",len(test_drivers), "Количество строк: ",len(X_test))


#масштабирование признаков
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test.values, dtype=torch.float32).reshape(-1, 1)


train_data = TensorDataset(X_train_t, y_train_t)
test_data = TensorDataset(X_test_t, y_test_t)

#рекомендуют для датасета 10к-100к батчсайз 64
batch_size = 64

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)


Количество водителей в тренировке:  247 Количество строк:  8465
Количество водителей в тесте:  107 Количество строк:  4063


In [5]:
def seed_everything(seed):
    random.seed(seed) # фиксируем генератор случайных чисел
    os.environ['PYTHONHASHSEED'] = str(seed) # фиксируем заполнения хешей
    np.random.seed(seed) # фиксируем генератор случайных чисел numpy
    torch.manual_seed(seed) # фиксируем генератор случайных чисел pytorch
    torch.cuda.manual_seed(seed) # фиксируем генератор случайных чисел для GPU
    #torch.backends.cudnn.deterministic = True # выбираем только детерминированные алгоритмы (для сверток)
    #torch.backends.cudnn.benchmark = False # фиксируем алгоритм вычисления сверток
seed_everything(30)

In [31]:
# функция потерь
criterion =  nn.PoissonNLLLoss(log_input=False)

In [32]:

# функция обучения модели
def train(model, device, train_loader, optimizer, criterion, epoch):
    model.train() # обязательно переводим в режим обучения
    train_loss_sum = 0

    n_ex = len(train_loader)

    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device) # переводим картинки и таргеты на GPU
        # обнуляем градиенты!
        optimizer.zero_grad()
        # прямой проход
        output = model(data)

        train_loss = criterion(output, target) # считаем значение функции потерь
        # обратный проход
        train_loss.backward()
        # делаем градиентный шаг оптимизатором
        optimizer.step()
        # считаем метрики и лосс
        train_loss_sum += train_loss.item() * data.size(0)


# функция тестирования
def test(model, device, test_loader, criterion):
    model.eval() # переводем модель в режим инференса
    test_loss_sum = 0

    all_predictions = []
    all_targets = []

    # показываем, что обученич нет и градиенты не обновляются
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss = criterion(output, target) # считаем значение функции потерь
            test_loss_sum += test_loss.item() * data.size(0)

            predictions = output.cpu().numpy().flatten()
            targets = target.cpu().numpy().flatten()

            all_predictions.extend(predictions.tolist())
            all_targets.extend(targets.tolist())


            # считаем метрики
    return test_loss_sum / len(test_loader.dataset)

In [33]:
class CFG:

    feature_cols = [
        'engine_capacity', 'road_quality_moderate', 'slope_flat',
        'motorway', 'rural', 'more_than_one_lane', 'congested',
        'speed_limit_mean', 'weather_temperature_mean', 'total_distance',
        'sum_roundabout', 'sum_traffic_signal', 'sum_stop_sign',
        'sum_yield_sign', 'sum_pedestrian_crossing_sign',
        'sum_animal_crossing_sign', 'speeding_serious',
        'harsh_acceleration', 'harsh_braking'
    ]

    input_dim = len(feature_cols)

    hidden_dims = [32,16,8]
    dropout_rate = 0.2
    use_dropout = True
    use_batchnorm = False
    activation = "relu"
    learning_rate = 0.001
    num_epochs = 60

In [34]:
def get_activation(name):

    activations = {
        "relu": nn.ReLU(),
        "leakyrelu": nn.LeakyReLU(),
        "gelu": nn.GELU(),
        "elu": nn.ELU(),
        "tanh": nn.Tanh(),
        "sigmoid": nn.Sigmoid()}

    return activations[name]

In [35]:
class Regression(nn.Module): # наследуемся от класса nn.Module
    def __init__(self):
        super(Regression,self).__init__()
        # организуем 3 скрытых слоя
        hidden_1 =  CFG.hidden_dims[0]
        hidden_2 = CFG.hidden_dims[1]
        hidden_3 = CFG.hidden_dims[2]

        input_dim = 19

        layers = []

        layers.append(nn.Linear(input_dim, hidden_1))
        if CFG.use_batchnorm:
            layers.append(nn.BatchNorm1d(hidden_1))
        layers.append(get_activation(CFG.activation))
        if CFG.use_dropout:
            layers.append(nn.Dropout(CFG.dropout_rate))

        layers.append(nn.Linear(hidden_1, hidden_2))
        if CFG.use_batchnorm:
            layers.append(nn.BatchNorm1d(hidden_2))
        layers.append(get_activation(CFG.activation))
        if CFG.use_dropout:
            layers.append(nn.Dropout(CFG.dropout_rate))

        layers.append(nn.Linear(hidden_2, hidden_3))
        if CFG.use_batchnorm:
            layers.append(nn.BatchNorm1d(hidden_3))
        layers.append(get_activation(CFG.activation))
        if CFG.use_dropout:
            layers.append(nn.Dropout(CFG.dropout_rate))

        layers.append(nn.Linear(hidden_3,1))

        self.net = nn.Sequential(*layers)

    def forward(self,x):
        x = torch.exp(self.net(x))
        return x

In [36]:
from adan_pytorch import Adan
import torch_optimizer as optim

def get_optimizer(name, model):
    if name== "Adam":
        return torch.optim.Adam(model.parameters(),lr=0.001)
    elif name== "Yogi":
        return optim.Yogi(model.parameters(),lr=0.01,betas=(0.9,0.999),eps=1e-3,weight_decay=0)
    elif name== "Adan":
        return Adan(model.parameters(),lr=0.01,betas=(0.98,0.92,0.99),weight_decay=0.02,eps=1e-8)

In [37]:
def main(model, optimizer):

    seed_everything(30)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # выделили устройство
    model = model.to(device)
    losses = []

    for epoch in range(1, CFG.num_epochs + 1): # цикл на эпохи
        train(model, device, train_loader, optimizer, criterion, epoch)
        losses.append(test(model, device, test_loader, criterion))

    best_epoch = np.argmin(losses)+1

    return min(losses), best_epoch

In [38]:
def grid_search():
    best_loss = float('inf')
    best_params = None

    hidden_variants = [
        [32, 16, 8],
        [64, 32, 16],
        [128, 64, 32],
  #      [256, 128, 64]
    ]

    activations = [
        "relu",
       # "leakyrelu",
        "gelu",
      #  "elu",
        "tanh",
      #  "sigmoid"
    ]

    optimizers = [
        "Adam",
       # "SGD_0001",
        #"SGD_001",
       # "RMSprop",
       # "Adamax",
       # "NAdam",
        #"AdamW",
        "Yogi",
        "Adan"
    ]

    experiment_num = 0

    for hidden_dims in hidden_variants:
        for use_dropout in [True, False]:
            for use_batchnorm in [True, False]:
                for activation in activations:
                    for optimizer_name in optimizers:

                        experiment_num += 1

                        CFG.hidden_dims = hidden_dims
                        CFG.use_dropout = use_dropout
                        CFG.use_batchnorm = use_batchnorm
                        CFG.activation = activation

                        model = Regression()

                        optimizer = get_optimizer(optimizer_name, model)
                        loss, best_epoch = main(model,optimizer)

                        if loss < best_loss:
                            best_loss = loss

                            best_params = {
                                "hidden_dims": hidden_dims,
                                "dropout": use_dropout,
                                "batchnorm": use_batchnorm,
                                "activation": activation,
                                "optimizer": optimizer_name,
                                "best_epoch": best_epoch}

                            print()
                            print("NEW BEST RESULT ", experiment_num)
                            print("Loss        :", best_loss)
                            print("Epoch       :", best_epoch)
                            print("Layers      :", hidden_dims)
                            print("Activation  :", activation)
                            print("Optimizer   :", optimizer_name)
                            print("Dropout     :", use_dropout)
                            print("BatchNorm   :", use_batchnorm)
                            print()

    return best_params, best_loss

In [39]:
best_params, best_loss = grid_search()

print("\n\nFINAL BEST CONFIGURATION")
print(best_params)
print("BEST LOSS:", best_loss)


NEW BEST RESULT  1
Loss        : 0.4744944529722044
Epoch       : 20
Layers      : [32, 16, 8]
Activation  : relu
Optimizer   : Adam
Dropout     : True
BatchNorm   : True


NEW BEST RESULT  2
Loss        : 0.4730498960620806
Epoch       : 3
Layers      : [32, 16, 8]
Activation  : relu
Optimizer   : Yogi
Dropout     : True
BatchNorm   : True


NEW BEST RESULT  3
Loss        : 0.46836420858827627
Epoch       : 50
Layers      : [32, 16, 8]
Activation  : relu
Optimizer   : Adan
Dropout     : True
BatchNorm   : True


NEW BEST RESULT  6
Loss        : 0.4648904357011055
Epoch       : 36
Layers      : [32, 16, 8]
Activation  : gelu
Optimizer   : Adan
Dropout     : True
BatchNorm   : True


NEW BEST RESULT  9
Loss        : 0.4641705354079219
Epoch       : 36
Layers      : [32, 16, 8]
Activation  : tanh
Optimizer   : Adan
Dropout     : True
BatchNorm   : True



FINAL BEST CONFIGURATION
{'hidden_dims': [32, 16, 8], 'dropout': True, 'batchnorm': True, 'activation': 'tanh', 'optimizer': 'Adan', 

In [ ]:
NEW BEST RESULT  9
Loss        : 0.4641705354079219
Epoch       : 36
Layers      : [32, 16, 8]
Activation  : tanh
Optimizer   : Adan
Dropout     : True
BatchNorm   : True



FINAL BEST CONFIGURATION
{'hidden_dims': [32, 16, 8], 'dropout': True, 'batchnorm': True, 'activation': 'tanh', 'optimizer': 'Adan', 'best_epoch': np.int64(36)}
BEST LOSS: 0.4641705354079219

In [12]:


class CFG:

# Задаем параметры нашего эксперимента
  feature_cols = ['engine_capacity', 'road_quality_moderate', 'slope_flat',
                'motorway', 'rural', 'more_than_one_lane', 'congested',
                'speed_limit_mean', 'weather_temperature_mean', 'total_distance',
                'sum_roundabout', 'sum_traffic_signal', 'sum_stop_sign',
                'sum_yield_sign', 'sum_pedestrian_crossing_sign',
                'sum_animal_crossing_sign', 'speeding_serious',
                'harsh_acceleration', 'harsh_braking']

  input_dim = len(feature_cols)
  hidden_dims = [32, 16, 8]
  dropout_rate = 0.2
  learning_rate = 0.001
  num_epochs = 60


class Regression(nn.Module): # наследуемся от класса nn.Module
    def __init__(self):
        super(Regression,self).__init__()
        # организуем 3 скрытых слоя
        hidden_1 =  CFG.hidden_dims[0]
        hidden_2 = CFG.hidden_dims[1]
        hidden_3 = CFG.hidden_dims[2]
        #
        input_dim = 19
        self.net = torch.nn.Sequential(
                      torch.nn.Linear(input_dim, hidden_1),
                      torch.nn.BatchNorm1d(hidden_1),
                      torch.nn.Tanh(),
                      torch.nn.Dropout(CFG.dropout_rate),
                      torch.nn.Linear(hidden_1, hidden_2),
                      torch.nn.BatchNorm1d(hidden_2),
                      torch.nn.Tanh(),
                      torch.nn.Dropout(CFG.dropout_rate),
                      torch.nn.Linear(hidden_2, hidden_3),
                      torch.nn.BatchNorm1d(hidden_3),
                      torch.nn.Tanh(),
                      torch.nn.Dropout(CFG.dropout_rate),
                      torch.nn.Linear(hidden_3, 1),
                    )

    def forward(self,x):
        x = torch.exp(self.net(x))
        return x

model = Regression()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device) # переводим модель на GPU, не получилось, поэтому CPU
print(model) # посмотрим на нашу модель

# функция потерь
criterion =  nn.PoissonNLLLoss(log_input=False)

optimizer = Adan(model.parameters(),lr=0.01,betas=(0.98,0.92,0.99),weight_decay=0.02,eps=1e-8)

# функция обучения модели
def train(model, device, train_loader, optimizer, criterion, epoch):
    model.train() # обязательно переводим в режим обучения
    train_loss_sum = 0

    n_ex = len(train_loader)

    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device) # переводим картинки и таргеты на GPU
        # обнуляем градиенты!
        optimizer.zero_grad()
        # прямой проход
        output = model(data)

        train_loss = criterion(output, target) # считаем значение функции потерь
        # обратный проход
        train_loss.backward()
        # делаем градиентный шаг оптимизатором
        optimizer.step()
        # считаем метрики и лосс
        train_loss_sum += train_loss.item() * data.size(0)


# функция тестирования
def test(model, device, test_loader, criterion):
    model.eval() # переводем модель в режим инференса
    test_loss_sum = 0

    all_predictions = []
    all_targets = []

    # показываем, что обученич нет и градиенты не обновляются
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss = criterion(output, target) # считаем значение функции потерь
            test_loss_sum += test_loss.item() * data.size(0)

            predictions = output.cpu().numpy().flatten()
            targets = target.cpu().numpy().flatten()

            all_predictions.extend(predictions.tolist())
            all_targets.extend(targets.tolist())


            # считаем метрики
    return test_loss_sum / len(test_loader.dataset)

# основная функция для экспериментов
def main(model):

    seed_everything(30)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # выделили устройство
    model = model.to(device)

    losses = []
    for epoch in range(1, CFG.num_epochs + 1): # цикл на эпохи

        train(model, device, train_loader, optimizer, criterion, epoch)
        losses.append(test(model, device, test_loader, criterion))
    print('Training is ended!')
    best_epoch = np.argmin(losses)+1
    print()
    print('BEST EPOCH: ', best_epoch, 'with Loss: ',min(losses))
main(model)

Regression(
  (net): Sequential(
    (0): Linear(in_features=19, out_features=32, bias=True)
    (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): Tanh()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=32, out_features=16, bias=True)
    (5): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): Tanh()
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_features=16, out_features=8, bias=True)
    (9): BatchNorm1d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): Tanh()
    (11): Dropout(p=0.2, inplace=False)
    (12): Linear(in_features=8, out_features=1, bias=True)
  )
)
Training is ended!

BEST EPOCH:  36 with Loss:  0.4651999700261606


Добавим еще слой

In [13]:


class CFG:

# Задаем параметры нашего эксперимента
  feature_cols = ['engine_capacity', 'road_quality_moderate', 'slope_flat',
                'motorway', 'rural', 'more_than_one_lane', 'congested',
                'speed_limit_mean', 'weather_temperature_mean', 'total_distance',
                'sum_roundabout', 'sum_traffic_signal', 'sum_stop_sign',
                'sum_yield_sign', 'sum_pedestrian_crossing_sign',
                'sum_animal_crossing_sign', 'speeding_serious',
                'harsh_acceleration', 'harsh_braking']

  input_dim = len(feature_cols)
  hidden_dims = [32, 16, 8, 4]
  dropout_rate = 0.2
  learning_rate = 0.001
  num_epochs = 60


class Regression(nn.Module): # наследуемся от класса nn.Module
    def __init__(self):
        super(Regression,self).__init__()
        # организуем 3 скрытых слоя
        hidden_1 =  CFG.hidden_dims[0]
        hidden_2 = CFG.hidden_dims[1]
        hidden_3 = CFG.hidden_dims[2]
        hidden_4 = CFG.hidden_dims[3]
        #
        input_dim = 19
        self.net = torch.nn.Sequential(
                      torch.nn.Linear(input_dim, hidden_1),
                      torch.nn.BatchNorm1d(hidden_1),
                      torch.nn.Tanh(),
                      torch.nn.Dropout(CFG.dropout_rate),
                      torch.nn.Linear(hidden_1, hidden_2),
                      torch.nn.BatchNorm1d(hidden_2),
                      torch.nn.Tanh(),
                      torch.nn.Dropout(CFG.dropout_rate),
                      torch.nn.Linear(hidden_2, hidden_3),
                      torch.nn.BatchNorm1d(hidden_3),
                      torch.nn.Tanh(),
                      torch.nn.Dropout(CFG.dropout_rate),
                      torch.nn.Linear(hidden_3, hidden_4),
                      torch.nn.BatchNorm1d(hidden_4),
                      torch.nn.Tanh(),
                      torch.nn.Dropout(CFG.dropout_rate),
                      torch.nn.Linear(hidden_4, 1),
                    )

    def forward(self,x):
        x = torch.exp(self.net(x))
        return x

model = Regression()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device) # переводим модель на GPU, не получилось, поэтому CPU
print(model) # посмотрим на нашу модель

# функция потерь
criterion =  nn.PoissonNLLLoss(log_input=False)

optimizer = Adan(model.parameters(),lr=0.01,betas=(0.98,0.92,0.99),weight_decay=0.02,eps=1e-8)

# функция обучения модели
def train(model, device, train_loader, optimizer, criterion, epoch):
    model.train() # обязательно переводим в режим обучения
    train_loss_sum = 0

    n_ex = len(train_loader)

    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device) # переводим картинки и таргеты на GPU
        # обнуляем градиенты!
        optimizer.zero_grad()
        # прямой проход
        output = model(data)

        train_loss = criterion(output, target) # считаем значение функции потерь
        # обратный проход
        train_loss.backward()
        # делаем градиентный шаг оптимизатором
        optimizer.step()
        # считаем метрики и лосс
        train_loss_sum += train_loss.item() * data.size(0)


# функция тестирования
def test(model, device, test_loader, criterion):
    model.eval() # переводем модель в режим инференса
    test_loss_sum = 0

    all_predictions = []
    all_targets = []

    # показываем, что обученич нет и градиенты не обновляются
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss = criterion(output, target) # считаем значение функции потерь
            test_loss_sum += test_loss.item() * data.size(0)

            predictions = output.cpu().numpy().flatten()
            targets = target.cpu().numpy().flatten()

            all_predictions.extend(predictions.tolist())
            all_targets.extend(targets.tolist())


            # считаем метрики
    return test_loss_sum / len(test_loader.dataset)

# основная функция для экспериментов
def main(model):

    seed_everything(30)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # выделили устройство
    model = model.to(device)

    losses = []
    for epoch in range(1, CFG.num_epochs + 1): # цикл на эпохи

        train(model, device, train_loader, optimizer, criterion, epoch)
        losses.append(test(model, device, test_loader, criterion))
    print('Training is ended!')
    best_epoch = np.argmin(losses)+1
    print()
    print('BEST EPOCH: ', best_epoch, 'with Loss: ',min(losses))
main(model)

Regression(
  (net): Sequential(
    (0): Linear(in_features=19, out_features=32, bias=True)
    (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): Tanh()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=32, out_features=16, bias=True)
    (5): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): Tanh()
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_features=16, out_features=8, bias=True)
    (9): BatchNorm1d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): Tanh()
    (11): Dropout(p=0.2, inplace=False)
    (12): Linear(in_features=8, out_features=4, bias=True)
    (13): BatchNorm1d(4, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (14): Tanh()
    (15): Dropout(p=0.2, inplace=False)
    (16): Linear(in_features=4, out_features=1, bias=True)
  )
)
Training is ended!

BEST EPOCH:  37 with Loss:  0.46813469465226926


дополнительный слой не улучшил ошибку